# 5주차 인공지능 기반 이상 탐지
정답 없이 웹 요청의 행동 특성을 학습하고, 결과를 조사할 이상 후보로 해석합니다.

## STEP 0. 실습 목표와 수업 진행
1. Nginx 원본 로그와 Feature CSV의 관계 이해
2. 정답을 제외한 12개 숫자형 특성으로 Isolation Forest 학습
3. 예측·점수·상위 후보 확인 및 교육용 정답과 사후 비교
4. contamination에 따른 경보량 비교
5. 오탐·미탐과 실제 운영 한계 분석

**필수:** STEP 1~16을 약 30분 동안 진행합니다. STEP 17~19와 학생 문제는 복습·과제로 활용합니다.
이 실습은 혼합 데이터에서 학습한 뒤 같은 데이터의 이상 후보를 찾는 **Outlier Detection**입니다.
아래 평가는 현재 합성 표본 안의 탐지 결과이며, 미래 로그에 대한 일반화 성능 평가가 아닙니다.
정답 `is_anomaly_truth`와 `anomaly_type`은 fit·특성 선택에 넣지 않습니다.

## Colab 준비
이 Notebook을 Colab에서 열고 **`nginx_access_sample.log` 한 개만** 준비하세요.
기존 CSV나 `generate_anomaly_dataset.py`를 업로드할 필요는 없습니다.

**실행 흐름:** 원본 로그 업로드 → 정규표현식 파싱 → DataFrame → 1분/5분 파생 특성 → CSV 생성 → 정답 없이 모델 학습 → 이상 후보 조사·사후 평가.
생성 프로그램의 `parse_log()`·`derive_features()` 계산을 아래에서 직접 실행합니다. 원본 로그를 재생성하거나 덮어쓰지 않습니다.
교육용 시나리오 정답은 결과 확인을 위해 별도로 연결하며 모델 학습에는 사용하지 않습니다.
파일은 현재 세션에만 업로드되므로 런타임 초기화 후 다시 업로드할 수 있습니다.
추가 설치나 Google Drive 마운트는 필요하지 않습니다. 로컬 실행에서는 Notebook과 원본 로그를 같은 폴더에 둡니다. CSV는 실행 중 생성됩니다.

## STEP 1. 라이브러리 — 필수 실습

**학습 목표:** 실습 도구를 준비합니다.

**배경:** pandas는 표, matplotlib는 그래프, scikit-learn은 모델을 담당합니다.

**결과 확인:** 버전 출력과 오류 여부를 확인하세요.

In [ ]:
# 파일 확인, 정규표현식 파싱, 원본 식별, 최근 요청 목록 관리용 표준 도구입니다.
import os
import re
import hashlib
from pathlib import Path
from collections import deque

# 표 분석과 시각화에 사용하는 라이브러리입니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 이상 탐지 모델, 거리 기반 모델의 스케일링, 사후 평가 도구입니다.
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)
import sklearn
# Colab 기본 폰트에서도 그래프가 읽히도록 축과 제목은 영문을 사용합니다.
plt.rcParams['figure.figsize'] = (8, 4)
print('pandas:', pd.__version__, 'scikit-learn:', sklearn.__version__)

## STEP 2. Nginx 원본 로그 업로드 — 필수 실습

**학습 목표:** 기존 CSV 대신 원본 웹 요청 로그부터 분석을 시작한다.

**실습 방법:** 아래 `files.upload()` 창에서 배포한 `nginx_access_sample.log`를 선택한다.
원본이 현재 런타임에 있으면 업로드를 생략한다. CSV와 생성 프로그램은 업로드하지 않는다.

**결과 확인:** 파일명은 정확히 `nginx_access_sample.log`이다.
이후 `web_anomaly_dataset.csv`를 현재 작업 폴더에 저장한다. 같은 이름의 기존 CSV가 있다면 갱신된다.

In [ ]:
RAW_FILE = 'nginx_access_sample.log'
CSV_FILE = 'web_anomaly_dataset.csv'

# 원본 파일이 이미 있으면 재업로드를 생략합니다.
if not os.path.exists(RAW_FILE):
    try:
        from google.colab import files
    except ImportError:
        # 로컬 Jupyter에서는 파일 선택 창 대신 같은 폴더의 원본을 사용합니다.
        print('로컬 실행: Notebook과 같은 폴더에 nginx_access_sample.log를 두세요.')
    else:
        uploaded = files.upload()
        print('업로드된 파일:', list(uploaded.keys()))

# 교수자 PC의 Windows 경로 대신 현재 런타임에 있는 파일명으로 접근합니다.
assert os.path.exists(RAW_FILE), 'nginx_access_sample.log를 업로드하고 이 셀을 다시 실행하세요.'
print('원본 파일:', RAW_FILE)

## STEP 3-A. 원본 웹 로그 직접 읽기 — 필수 실습

**목적:** CSV의 값이 어디에서 추출되는지 실제 로그 한 줄로 확인한다.

이 배포본은 **Nginx Combined 형식 + 사용자 정의 `rt=$request_time`**을 따르는 합성 로그이다.
IP, 시각, HTTP 요청, 상태 코드, 응답 본문 크기, referer, user-agent, 처리 시간을 담는다.
`rt`는 초 단위 요청 처리 시간이며, 기본 Combined 형식에 별도로 추가한 값이다.

**읽는 예:** `200 3396 ... rt=0.012`는 HTTP 상태 200, 응답 본문 3,396바이트,
요청 처리 시간 0.012초이다. 성공적인 로그인이나 사용자 신원까지 증명하는 값은 아니다.

In [ ]:
# 파일 전체를 읽고 줄바꿈을 기준으로 요청 기록을 나눕니다.
# text 모드 읽기는 Windows/Colab의 줄바꿈 차이도 표준화합니다.
raw_text = Path(RAW_FILE).read_text(encoding='utf-8')
lines = raw_text.splitlines()
print('원본 요청 수:', len(lines))
print('\n'.join(lines[:3]))

## STEP 3-B. 정규표현식으로 필드 추출하기 — 필수 실습

**목적:** 로그 문자열을 열 이름과 자료형이 있는 DataFrame으로 바꾼다.

- `(?P<client_ip>...)`처럼 이름을 붙인 그룹이 각 필드의 값을 추출한다.
- `fullmatch()`는 한 줄 전체가 배포 형식과 맞는지 검사한다. 실패한 줄은 번호와 함께 보고한다.
- `groupdict()`는 추출 결과를 `{열 이름: 값}` 사전으로 만든다.
- 시각은 시간대가 포함된 datetime, 상태·응답 크기는 정수, rt는 실수로 변환한다.
- `raw_file`·`line_no`·`event_id`를 남겨 분석 행에서 원본 요청으로 되돌아간다.

**시각 형식:** `%d/%b/%Y:%H:%M:%S %z`는 `14/Oct/2026:02:00:00 +0900`에 대응한다.
`+0900`은 이미 표시된 시각의 UTC 오프셋이다. 9시간을 다시 더하지 않는다.
이 파서는 배포본의 고정 형식용이며 모든 Nginx log_format에 공통 적용되는 파서는 아니다.

In [ ]:
# 로그 구조에 맞춰 이름 있는 그룹을 정의합니다.
# \S+는 공백 없는 값, [^\]]+는 ] 전까지의 시각, \d{3}은 상태 코드 3자리입니다.
PATTERN = re.compile(
    r'(?P<client_ip>\S+) - - \[(?P<timestamp>[^\]]+)\] '
    r'"(?P<method>\S+) (?P<uri>\S+) (?P<http_version>[^" ]+)" '
    r'(?P<status_code>\d{3}) (?P<body_bytes_sent>\d+) '
    r'"(?P<referer>[^"]*)" "(?P<user_agent>[^"]*)" rt=(?P<request_time>[\d.]+)'
)

def parse_log(path):
    path = Path(path)
    records = []  # 요청별로 추출한 사전을 보관합니다.
    for line_no, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        match = PATTERN.fullmatch(line)
        if not match:
            # 틀린 행을 조용히 버리지 않고 원본 파일과 줄 번호를 알려 줍니다.
            raise ValueError(f'파싱 실패: {path.name}:{line_no}')
        row = match.groupdict()
        # 시간대가 포함된 시각을 datetime으로 바꿉니다. KST로 표현을 통일합니다.
        row['timestamp'] = pd.to_datetime(
            row['timestamp'], format='%d/%b/%Y:%H:%M:%S %z'
        ).tz_convert('Asia/Seoul')
        for col in ['status_code', 'body_bytes_sent']:
            row[col] = int(row[col])
        row['request_time'] = float(row['request_time'])
        # WEB00001 등의 ID는 원본 행 추적용으로 추가한 값이며 Nginx 원본 필드는 아닙니다.
        row.update(raw_file=path.name, line_no=line_no, event_id=f'WEB{line_no:05d}')
        records.append(row)
    if not records:
        raise ValueError('원본 로그에 요청 기록이 없습니다.')
    return pd.DataFrame(records)

parsed_df = parse_log(RAW_FILE)
print('파싱 결과:', parsed_df.shape)  # 원본 관측 10열 + 추적 3열
display(parsed_df[['event_id', 'timestamp', 'client_ip', 'uri',
                   'status_code', 'body_bytes_sent', 'request_time']].head())
display(parsed_df[['timestamp', 'status_code', 'body_bytes_sent', 'request_time']].dtypes)

## STEP 3-C. IP별 최근 1분·5분 파생 특성 계산 — 필수 실습

**목적:** 요청 한 건의 값에 반복·경로 다양성·오류·평균 전송량 같은 행동 정보를 더한다.

- 기준은 현재 행의 요청 완료 로그 시각 `timestamp=t`이다.
- 1분 범위는 **(t−60초, t]**, 5분 범위는 **(t−300초, t]**이다.
- 현재 행을 포함하고 정확히 60초/300초 전 행은 제외한다. 미래 행은 사용하지 않는다.
- 같은 시각의 요청은 `line_no` 순서로 현재 행까지만 포함한다.
- `deque`는 해당 IP의 최근 요청을 보관한다. 오래된 요청을 제거하고 현재 요청을 추가한다.

| 파생 특성 | 계산 내용 |
|---|---|
| hour / is_night | 한국 시간의 시 / 00:00~05:59이면 1 |
| requests_1m / requests_5m | 같은 IP의 최근 1분 / 5분 요청 수 |
| unique_paths_5m | 같은 IP의 최근 5분 서로 다른 URI 수(쿼리 포함) |
| error_count_5m / error_rate_5m | 4xx·5xx 요청 수 / 5분 요청 수로 나눈 비율 |
| same_path_count_1m | 같은 IP의 최근 1분 중 현재 URI와 같은 요청 수 |
| avg_bytes_5m / avg_request_time_5m | 같은 IP의 최근 5분 평균 응답 크기 / 처리 시간 |
| is_admin_path | 쿼리 앞 경로가 /admin 또는 /management로 시작하면 1 |

**작은 예:** 같은 IP가 0초 `/login` 200, 20초 `/admin` 404,
40초 `/login` 200을 요청했다면 40초 행은 요청 수=3, URI 종류=2,
오류율=1/3, 같은 URI 요청 수=2이다. 60초 행에서 0초 요청은 1분 범위에서 제외된다.

IP는 사용자와 일대일로 대응하지 않는다. 관리자 경로 접근과 HTTP 200도 인증 성공을 확정하지 않는다.
응답 크기·rt는 완료 후의 값이므로 이 실습은 완료된 요청 로그의 분석이다.

In [ ]:
def derive_features(frame):
    # 시각이 같은 요청은 원본 줄 번호 순서로 정렬해 집계의 순서를 일정하게 합니다.
    result = frame.sort_values(['timestamp', 'line_no'], kind='stable').reset_index(drop=True).copy()
    history = {}  # {client_ip: 최근 5분 요청 deque}
    derived = []  # 각 원본 행과 같은 순서로 계산 결과를 저장합니다.

    for row in result.itertuples(index=False):
        t = row.timestamp.timestamp()  # 시간 비교를 위한 epoch 초
        q = history.setdefault(row.client_ip, deque())
        # 정확히 300초 전 기록도 제외하여 (t-300, t] 범위를 유지합니다.
        while q and q[0][0] <= t - 300:
            q.popleft()
        # 저장 항목: 시각, URI, 상태 코드, 응답 크기, 처리 시간
        q.append((t, row.uri, row.status_code, row.body_bytes_sent, row.request_time))

        # 5분 기록 중 최근 1분에 해당하는 요청만 선택합니다.
        recent = [item for item in q if item[0] > t - 60]
        errors = sum(item[2] >= 400 for item in q)  # 제공한 유효 HTTP 코드에서 4xx·5xx

        # URI 종류·동일 URI는 쿼리까지 비교하되 관리자 판단은 쿼리를 제외합니다.
        admin = row.uri.split('?')[0].startswith(('/admin', '/management'))

        derived.append({
            'hour': row.timestamp.hour,
            'is_night': int(row.timestamp.hour < 6),
            'requests_1m': len(recent),
            'requests_5m': len(q),
            'unique_paths_5m': len({item[1] for item in q}),  # set으로 중복 URI 제거
            'error_count_5m': errors,
            'error_rate_5m': errors / len(q),  # 현재 요청이 있으므로 분모는 항상 1 이상
            'same_path_count_1m': sum(item[1] == row.uri for item in recent),
            'avg_bytes_5m': np.mean([item[3] for item in q]),
            'avg_request_time_5m': np.mean([item[4] for item in q]),
            'is_admin_path': int(admin)
        })
    # axis=1은 동일한 요청 행에 계산한 열을 가로로 붙이는 뜻입니다.
    return pd.concat([result, pd.DataFrame(derived)], axis=1)

feature_df = derive_features(parsed_df)
print('정답 연결 전:', feature_df.shape)  # 3,000행 × 24열
display(feature_df[['event_id', 'client_ip', 'uri', 'requests_1m',
                    'requests_5m', 'unique_paths_5m', 'error_rate_5m']].head())

## STEP 3-D. 사후 평가용 교육 정답 연결 — 필수 실습

**모델은 정답 없이 학습한다.** 이상 유형·정답은 모델 입력이 아니라 STEP 14 이후의
탐지 결과 확인을 위해 보관한다. 정답이 없어도 파싱·파생 특성 계산·모델 학습은 가능하다.

**이번 정답의 근거:** 생성 프로그램이 요청을 만들 때 알고 있던 시나리오 이력이다.
아래 `scenario_line_numbers`는 배포 원본의 각 줄과 그 시나리오를 연결한 교수자용 정답 표이다.
기존 CSV의 줄별 정답과 대조했으며, URI·상태 코드·요청 수를 보고 임의로 판단한 값이 아니다.
NORMAL 2,850건 / PATH_SCAN 50건 / REQUEST_BURST 50건 / LARGE_TRANSFER 30건 / ODD_HOUR_ADMIN 20건이다.

원본 지문을 확인한 다음 줄 번호로 정답을 붙인다. **줄 번호 자체에 정상·이상 의미는 없다.**
다른 로그에는 이 정답 표를 사용할 수 없다. 실제 운영에서는 정답이 없는 경우가 많아
담당자의 조사와 인증 로그·작업 승인 내역 등으로 후보를 확인한다.

In [ ]:
# 각 목록은 해당 시나리오의 원본 줄 번호입니다. 관측값에서 추정한 분류 규칙이 아닙니다.
scenario_line_numbers = {'PATH_SCAN': [1471, 1472, 1473, 1474, 1475, 1476, 1477, 1478, 1479, 1480, 1481, 1482, 1483, 1484,
               1485, 1486, 1487, 1488, 1489, 1490, 1492, 1493, 1494, 1495, 1496, 1497, 1498, 1499,
               1501, 1502, 1503, 1504, 1505, 1506, 1508, 1509, 1510, 1511, 1512, 1513, 1514, 1515,
               1517, 1518, 1519, 1520, 1521, 1522, 1523, 1524],
 'REQUEST_BURST': [1809, 1810, 1811, 1812, 1813, 1814, 1815, 1816, 1817, 1818, 1819, 1820, 1821,
                   1822, 1823, 1824, 1825, 1826, 1827, 1828, 1829, 1830, 1831, 1832, 1833, 1834,
                   1835, 1836, 1837, 1838, 1839, 1840, 1841, 1842, 1843, 1844, 1845, 1846, 1847,
                   1848, 1849, 1850, 1851, 1852, 1853, 1855, 1856, 1857, 1858, 1859],
 'LARGE_TRANSFER': [2122, 2125, 2129, 2130, 2132, 2133, 2135, 2139, 2141, 2143, 2144, 2146, 2150,
                    2154, 2156, 2159, 2162, 2164, 2166, 2168, 2169, 2174, 2175, 2182, 2183, 2187,
                    2189, 2191, 2193, 2197],
 'ODD_HOUR_ADMIN': [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]}
teaching_truth = pd.DataFrame({
    'line_no': range(1, 3001),
    'anomaly_type': ['NORMAL'] * 3000
})

for label, numbers in scenario_line_numbers.items():
    # isin()은 지정된 줄 번호에 해당하는 행만 True로 표시합니다.
    teaching_truth.loc[teaching_truth['line_no'].isin(numbers), 'anomaly_type'] = label

# 같은 줄이 서로 다른 시나리오에 중복 기재되지 않았는지 확인합니다.
all_anomaly_lines = [number for numbers in scenario_line_numbers.values() for number in numbers]

# 1:1 검증으로 의도하지 않은 행 증식을 막으며 시각순 행을 그대로 유지합니다.
dataset = feature_df.merge(teaching_truth, on='line_no', how='left', validate='one_to_one')
dataset['is_anomaly_truth'] = dataset['anomaly_type'].ne('NORMAL').astype(int)
print('정답은 사후 평가용입니다. 모델 fit에는 전달하지 않습니다.')

## STEP 3-E. 직접 생성한 CSV 저장·읽기 — 필수 실습

**목적:** 원본 관측 10열 + 추적 3열 + 파생 특성 11열 + 사후 평가용 정답 2열을 저장한다.

`index=False`는 DataFrame의 행 번호를 CSV 열로 추가하지 않게 한다.
CSV에는 원래 로그의 rt가 `request_time`, 응답 본문 크기가 `body_bytes_sent`로 저장된다.
생성 파일명은 `web_anomaly_dataset.csv`이며 기존과 동일한 **3,000행 × 26열**이다.

`pd.read_csv()`로 다시 읽은 시각은 문자열이므로 datetime·한국 시간으로 복원한다.
`head()`·`shape`·`info()`로 값·크기·자료형을 확인한다.
Colab 왼쪽 파일 목록에서 CSV를 다운로드할 수 있다.

In [ ]:
# 배포 시나리오의 구조와 수치 품질을 확인한 후 CSV만 저장합니다. 원본은 수정하지 않습니다.
dataset.to_csv(CSV_FILE, index=False, encoding='utf-8')
print('CSV 생성 완료:', CSV_FILE, dataset.shape)

In [ ]:
# 방금 생성한 CSV를 다시 읽어 기존 이상 탐지 실습의 df 변수로 이어갑니다.
df = pd.read_csv(CSV_FILE)

# CSV의 시각 문자열을 시간대가 있는 datetime으로 복원합니다.
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_convert('Asia/Seoul')
display(df.head())
print('행·열:', df.shape)
df.info()

## STEP 4. 데이터 구조 확인 — 필수 실습

**학습 목표:** 누락·수치 범위·시각을 확인합니다.

**배경:** 잘못된 값이 모델의 이상 점수에 영향을 줄 수 있습니다.

**결과 확인:** 3,000행·26열, 결측치 0, KST 시간 순서인지 확인하세요.

In [ ]:
# 원본·추적·파생·정답 열을 구분하고 결측치·시각순 정렬을 점검합니다.
print('컬럼:', list(df.columns))
print('결측치 합계:', int(df.isna().sum().sum()))
print('시간 범위:', df['timestamp'].min(), '~', df['timestamp'].max())

# describe()는 숫자 열의 개수·평균·표준편차·최솟값·분위수·최댓값을 보여 줍니다.
display(df.describe().T)

## STEP 5. 원본 로그와 CSV 비교 — 필수 실습

**학습 목표:** 원본에서 가져온 값과 계산한 값을 구분합니다.

**배경:** STEP 3에서 원본을 파싱했습니다. 아래 한 줄은 같은 배포 로그의 실제 첫 번째 행입니다. rt는 사용자 정의 request time 필드입니다.

**결과 확인:** IP·URI·상태·응답 크기·rt를 CSV 첫 행과 비교하세요.

```text
192.0.2.91 - - [14/Oct/2026:02:00:00 +0900] "GET /admin/health HTTP/1.1" 200 3396 "https://example.com/" "Mozilla/5.0" rt=0.012
```

`raw_file`, `line_no`, `event_id`는 원본 추적용입니다. 로그 시각은 요청 처리 완료 시점입니다.

최근 1분/5분은 해당 행의 시각을 기준으로 과거와 현재만 사용합니다. 같은 시각은 원본 행 순서로 현재 행까지 포함합니다.

In [ ]:
# 같은 원본 줄의 관측값과 CSV 값을 대조합니다. event_id는 학습 입력이 아닙니다.
matched = df.set_index('event_id').loc[parsed_df['event_id']]
print('원본 → CSV 요청 3,000행 대조 완료')
display(df.loc[:0, ['raw_file','line_no','client_ip','uri','status_code','body_bytes_sent','request_time','requests_1m']])

## STEP 6. 교육용 정답 분포 확인 — 필수 실습

**학습 목표:** 생성 시나리오를 확인합니다.

**배경:** 실제 운영에서는 이런 정답이 없는 경우가 많지만 수업에서는 결과 확인을 위해 보관합니다.

**결과 확인:** NORMAL 2,850건, 이상 150건입니다. 이 정보로 모델을 학습하거나 입력을 고르지 않습니다.

In [ ]:
# 교육용 정답의 건수를 확인합니다. 이 값으로 입력 선택이나 모델 학습을 하지 않습니다.
display(df['anomaly_type'].value_counts())

# 0/1 정답의 평균은 1인 행의 비율입니다.
print('이상 비율:', df['is_anomaly_truth'].mean())

## STEP 7. 행동 특성 관찰 — 필수 실습

**학습 목표:** 값의 범위와 집계의 의미를 읽습니다.

**배경:** 한 요청만으로 알기 어려운 반복 행동을 requests_1m 등의 파생 특성으로 표현합니다.

**결과 확인:** 높은 요청 수가 항상 공격은 아닙니다. 정상 모니터링도 반복 요청을 만듭니다.

In [ ]:
# 값의 범위를 살펴봅니다. 높은 요청 수만으로 공격을 확정할 수 없습니다.
display(df[['requests_1m','unique_paths_5m','error_rate_5m',
            'body_bytes_sent','request_time','avg_bytes_5m','is_night']].describe().T)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(df['requests_1m'], bins=25, color='#375B88')
axes[0].set(xlabel='Requests in last 1 minute', ylabel='Requests')

# log10(bytes+1)은 0도 표현하면서 큰 응답 크기의 범위를 압축합니다. 원본 열은 유지합니다.
axes[1].hist(np.log10(df['body_bytes_sent'] + 1), bins=25, color='#375B88')
axes[1].set(xlabel='log10(response bytes + 1)', ylabel='Requests')
plt.tight_layout(); plt.show()

## STEP 8. 모델 입력 선택 — 필수 실습

**학습 목표:** 정답·식별자를 제외한 12개 숫자형 특성을 사용합니다.

**배경:** IP와 URI 원문은 추적용, method/status 등은 이번 기본 입력을 단순화하기 위해 제외합니다.

**결과 확인:** timestamp·client_ip·uri·referer·user_agent·두 정답 열이 X에 없어야 합니다.

In [ ]:
# 관측·파생 특성 12개만 지정합니다. 사후 평가 정답과 식별자는 포함하지 않습니다.
feature_columns = [
    'body_bytes_sent', 'request_time', 'hour', 'is_night',
    'requests_1m', 'requests_5m', 'unique_paths_5m', 'error_rate_5m',
    'same_path_count_1m', 'avg_bytes_5m', 'avg_request_time_5m', 'is_admin_path'
]

# 열을 명시적으로 선택해 정답과 식별자가 실수로 섞이지 않게 합니다.
X = df[feature_columns].copy()
print('모델 입력:', X.shape)
# Isolation Forest는 트리 기반이므로 이번 기본 실습은 Scaling 없이 진행합니다.
# LOF에서는 거리 계산을 위해 StandardScaler를 사용합니다.

## STEP 9. Isolation Forest 생성 — 필수 실습

**학습 목표:** 이상 후보를 찾는 모델을 설정합니다.

**배경:** 100개 트리를 만들고 contamination=0.05로 학습 표본의 임계값을 설정합니다.

**결과 확인:** 0.05는 운영자가 설정한 값입니다. 실제 정답의 비율을 알아냈다는 뜻은 아닙니다.

In [ ]:
model = IsolationForest(
    n_estimators=100,  # Isolation Tree 수
    contamination=0.05,  # 학습 점수에서 약 5%를 이상 후보로 구분할 임계값
    random_state=42  # 같은 환경에서 결과 재현
)

## STEP 10. 정답 없이 학습 — 필수 실습

**학습 목표:** 모델에 X만 제공합니다.

**배경:** 4주차 fit(X_train, y_train)과 달리 이번 fit에는 정답 y가 없습니다.

**결과 확인:** 비지도 학습도 입력 품질과 표본 구성에 의존합니다.

In [ ]:
# fit(X)는 입력 분포의 구조를 학습합니다. 정답 y는 전달하지 않습니다.
model.fit(X)

# 정답이 섞여 있는지 X를 다시 확인하세요. 여기서는 y를 전달하지 않습니다.
print('학습 완료:', len(model.estimators_), 'trees')

## STEP 11. 정상·이상 후보 예측 — 필수 실습

**학습 목표:** -1/1을 1/0으로 변환합니다.

**배경:** 모델 출력은 공격 확정이 아니라 표본 안의 이상 후보입니다.

**결과 확인:** predicted_anomaly=1인 행 수를 확인하세요.

In [ ]:
# sklearn 출력 규칙을 분석용 0/1 열로 바꾸되 공격 확정으로 해석하지 않습니다.
pred = model.predict(X)  # 1=Inlier, -1=Outlier
df['predicted_anomaly'] = (pred == -1).astype(int)  # 0=정상 후보, 1=이상 후보
display(df['predicted_anomaly'].value_counts().sort_index())

## STEP 12. 이상 점수와 상위 후보 — 필수 실습

**학습 목표:** 낮은 점수부터 조사할 기록을 확인합니다.

**배경:** decision_function은 score_samples-offset_입니다. 값이 낮을수록 더 이상하며, 0 미만은 Outlier입니다.

**결과 확인:** 점수는 공격 확률이 아닙니다. 상위 20행의 반복·오류·크기를 관찰하세요.

In [ ]:
# decision_function은 임계값 기준의 상대 점수이며 공격 확률이 아닙니다.
df['anomaly_score'] = model.decision_function(X)

review_columns = ['timestamp','client_ip','uri','status_code','requests_1m',
                  'unique_paths_5m','error_rate_5m','body_bytes_sent',
                  'anomaly_score','predicted_anomaly']

# 작은 점수부터 정렬하여 담당자가 우선 조사할 20개 요청을 표시합니다.
display(df.sort_values('anomaly_score')[review_columns].head(20))

## STEP 13. 결과 시각화 — 필수 실습

**학습 목표:** 모델이 선택한 후보를 비교합니다.

**배경:** 12차원 모델의 판단을 두 축으로 투영하므로 그래프에서 모든 차이가 보이지 않을 수 있습니다.

**결과 확인:** 같은 좌표라도 다른 특성으로 인해 예측이 달라질 수 있습니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# 예측 후보별로 나눠 표시합니다. 2개 축은 전체 12개 입력 중 일부만 보여 줍니다.
for flag, color, label in [(0, '#375B88', 'Inlier'), (1, '#D56A2A', 'Outlier')]:
    part = df[df['predicted_anomaly'].eq(flag)]
    axes[0].scatter(part['requests_1m'], part['unique_paths_5m'], s=16, alpha=.5, c=color, label=label)

axes[0].set(xlabel='requests_1m', ylabel='unique_paths_5m'); axes[0].legend()
axes[1].hist(df['anomaly_score'], bins=35, color='#375B88')

# 0은 decision_function의 후보 판정 경계입니다.
axes[1].axvline(0, color='#D56A2A', linestyle='--')
axes[1].set(xlabel='decision_function (lower = more unusual)', ylabel='Requests')
plt.tight_layout(); plt.show()

## STEP 14. 정답과 사후 비교 — 필수 실습

**학습 목표:** 현재 표본의 탐지 결과를 평가합니다.

**배경:** 이제 정답을 평가 함수에만 전달합니다. 정상만 예측해도 Accuracy는 95%가 됩니다.

**결과 확인:** 행=실제, 열=예측, 양성=ANOMALY로 CM의 FP/FN을 읽으세요.

In [ ]:
y_truth = df['is_anomaly_truth']  # 학습이 끝난 후 평가에만 사용합니다.
# 이상=1을 Positive로 평가합니다. zero_division=0은 분모가 0인 지표를 0으로 표시합니다.
def scores(truth, prediction):
    return {'Accuracy': accuracy_score(truth, prediction),
            'Precision': precision_score(truth, prediction, zero_division=0),
            'Recall': recall_score(truth, prediction, zero_division=0),
            'F1': f1_score(truth, prediction, zero_division=0)}
# 동일한 학습 표본의 사후 평가입니다. 미래 데이터 성능 검증과 구분합니다.
base_scores = scores(y_truth, df['predicted_anomaly'])
display(pd.DataFrame([base_scores], index=['Isolation Forest']))
# 행=실제 정답, 열=예측이며 [[TN, FP], [FN, TP]] 순서입니다.
cm = confusion_matrix(y_truth, df['predicted_anomaly'], labels=[0, 1])
ConfusionMatrixDisplay(cm, display_labels=['NORMAL','ANOMALY']).plot(cmap='Blues')
plt.title('Current-sample detection results'); plt.tight_layout(); plt.show()

**생각해보기:** 실제 이상인데 정상으로 예측한 칸은 어디인가요? 정상 95% 데이터에서 Accuracy만 보면 어떤 문제가 생기나요? 이 평가는 왜 미래 데이터 성능을 보장하지 않나요?

## STEP 15. 이상 유형별 탐지 — 필수 실습

**학습 목표:** 유형별로 탐지율이 다른지 확인합니다.

**배경:** 하나의 모델도 경로 탐색과 야간 관리자 접근을 같은 수준으로 찾지 못할 수 있습니다.

**결과 확인:** NORMAL의 탐지 비율은 오탐률, 이상 유형의 탐지 비율은 해당 유형의 Recall입니다.

In [ ]:
# size=전체 요청 수, sum=이상 후보 수, mean=해당 유형 내 후보 비율입니다.
type_results = df.groupby('anomaly_type')['predicted_anomaly'].agg(['size','sum','mean'])
type_results.columns = ['Count','Detected','Detection rate']
display(type_results)

## STEP 16. contamination 비교 — 필수 실습

**학습 목표:** 경보량과 Precision/Recall의 변화를 확인합니다.

**배경:** contamination은 학습 점수의 임계값에 영향을 줍니다. 기본 random_state와 다른 파라미터를 고정합니다.

**결과 확인:** 1%·3%·5%·10%의 탐지 수를 비교하되 이 표의 최고값으로 실제 운영 설정을 결정하지 않습니다.

In [ ]:
# 다른 설정은 유지하고 contamination만 바꿔 경보량 변화를 관찰합니다.
experiments = []
for fraction in [0.01, 0.03, 0.05, 0.10]:
    trial = IsolationForest(n_estimators=100, contamination=fraction, random_state=42)
    trial.fit(X)  # 정답은 사용하지 않습니다.
    trial_pred = (trial.predict(X) == -1).astype(int)
    row = {'contamination': fraction, 'Detected': int(trial_pred.sum())}
    # 정답은 학습 이후 비교에만 사용합니다. 이 표로 운영 설정을 확정하지 않습니다.
    row.update(scores(y_truth, trial_pred)); experiments.append(row)

comparison = pd.DataFrame(experiments)
display(comparison)
comparison.plot(x='contamination', y=['Precision','Recall','F1'], marker='o')
plt.ylabel('Score'); plt.ylim(0, 1.05); plt.tight_layout(); plt.show()

**운영 해석:** 더 큰 contamination은 이 학습 표본에서 더 많은 후보를 만듭니다. 미래 데이터의 이상 비율을 고정하지는 않습니다. 조사 인력·경보량·놓치면 안 되는 사건을 함께 고려하세요.

## STEP 17. False Positive 분석 — 추가 실습 OPTIONAL

**학습 목표:** 정상인데 이상으로 예측한 기록을 읽습니다.

**배경:** 승인된 자동화·대용량 다운로드도 모델에 드물게 보일 수 있습니다.

**결과 확인:** 운영자에게 점검 일정·업무 목적·계정 정보를 확인해야 합니다.

In [ ]:
# 정답은 정상(0)인데 모델 후보는 이상(1)인 행을 같은 표에서 복합 조건으로 선택합니다.
false_positive = df[(df['is_anomaly_truth'] == 0) & (df['predicted_anomaly'] == 1)]
print('False Positive:', len(false_positive))
display(false_positive.sort_values('anomaly_score')[review_columns].head(10))

## STEP 18. False Negative 분석 — 추가 실습 OPTIONAL

**학습 목표:** 실제 시나리오 이상인데 놓친 기록을 읽습니다.

**배경:** 패턴 초반에는 과거 요청이 적으며, 야간 정비와 이상 접속은 특성이 겹칠 수 있습니다.

**결과 확인:** 한 행만 보고 제거하지 말고 이전·이후 로그와 추가 정보를 확인하세요.

In [ ]:
# 정답은 시나리오 이상(1)인데 모델이 정상 후보(0)로 본 행을 선택합니다.
false_negative = df[(df['is_anomaly_truth'] == 1) & (df['predicted_anomaly'] == 0)]
print('False Negative:', len(false_negative))
display(false_negative[review_columns + ['anomaly_type']].head(10))

## STEP 19. LOF 비교 — 추가 실습 OPTIONAL

**학습 목표:** 주변 밀도 기반 방법과 비교합니다.

**배경:** LOF는 거리 영향을 받으므로 StandardScaler를 사용합니다. 이는 같은 표본의 outlier detection입니다.

**결과 확인:** novelty=False에서는 predict(X)가 아닌 fit_predict(X)를 사용합니다.

In [ ]:
# LOF의 거리 계산에서 단위 차이의 영향을 줄이도록 각 열을 표준화합니다.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
# 기본 novelty=False에서는 같은 표본에 fit_predict()를 사용합니다.
lof_pred = (lof.fit_predict(X_scaled) == -1).astype(int)
display(pd.DataFrame([base_scores, scores(y_truth, lof_pred)], index=['Isolation Forest','LOF']))
# 새로운 데이터에 LOF를 적용하는 novelty=True 설정은 이번 기본 실습 범위 밖입니다.

## 학생 수행 문제 — 7문제
1. STEP 16의 contamination 4개 값에서 경보량과 Precision/Recall을 비교하고, 조사 인력이 적을 때의 선택 근거를 쓰세요.
2. False Positive 최대 10개를 읽고, 승인된 정상 활동일 가능성과 추가 확인할 자료를 쓰세요.
3. `requests_1m`을 X에서 제외한 모델을 학습하고, 기본 모델과 Precision/Recall/F1을 비교하세요.
4. `unique_paths_5m`이 경로 탐색에 도움이 되는 이유를 설명하고, 이 특성을 제거한 결과를 확인하세요.
5. 이상 점수가 낮다는 사실만으로 공격을 확정할 수 없는 사례를 두 가지 쓰세요.
6. 4주차 지도학습 분류와 이번 비지도 이상 탐지의 정답 사용 방식·출력·한계를 비교하세요.
7. OPTIONAL: LOF와 Isolation Forest가 선택한 후보의 공통점과 차이를 찾아보세요.

### 추가 특성 실험의 실행 예
다음 코드는 문제 3을 수행하는 출발점입니다. 문제 4에서는 제거할 열 이름만 바꿉니다.

In [ ]:
# 원래 X는 유지하고 특성 하나를 제거한 복사본으로 새 모델을 학습합니다.
X_reduced = X.drop(columns=['requests_1m'])
reduced_model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
reduced_model.fit(X_reduced)
reduced_pred = (reduced_model.predict(X_reduced) == -1).astype(int)
display(pd.DataFrame([base_scores, scores(y_truth, reduced_pred)], index=['All features','Without requests_1m']))

## 종합 정리
완료된 웹 요청 로그를 파싱하고 현재·과거 행동을 집계한 뒤, 정답 없이 Isolation Forest를 학습했습니다.
점수와 후보를 조사하고 교육용 정답으로 현재 표본 안의 탐지 결과를 확인했습니다.
실제 운영에서는 미래 로그, 시간별 변화, 수집 누락, NAT/프록시, 경보 조사 인력을 고려해야 합니다.
IP는 사용자를 완전히 식별하지 않으며 URI 접근도 인증 성공을 증명하지 않습니다.

**이상 후보는 추가 조사의 출발점입니다. 공격 여부는 담당자가 다른 로그와 맥락을 확인하여 판단합니다.**

참고: [IsolationForest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html),
[Outlier/Novelty Detection](https://scikit-learn.org/stable/modules/outlier_detection.html),
[Nginx Log Module](https://nginx.org/en/docs/http/ngx_http_log_module.html)